# Lab 1: Linear Regression, from Algebra to Bayes

**ECON5129 Statistical Machine Learning** &middot; Adam Smith Business School, University of Glasgow

{{COLAB_BADGE}}

This lab accompanies Lecture 1. By the end of the session you should be able to:

1. Compute the least squares estimator three different ways and explain why the textbook formula is the wrong way to program it.
2. Interpret least squares as an orthogonal projection, and read the hat matrix, leverage and $R^2$ off that geometry.
3. Verify the sampling properties of $\widehat{\boldsymbol{\beta}}$ by simulation, and see through the singular value decomposition why collinearity inflates variance.
4. Fit nonlinear functions with a linear model through basis expansions, and watch overfitting appear.
5. Estimate the same model by maximum likelihood, and change the likelihood to change the estimator.
6. Derive the Bayesian posterior numerically, recognise the ridge estimator inside it, and produce predictive intervals.

**Plan for the session**

| Time | Part |
|---|---|
| 0:00 - 0:10 | Setup |
| 0:10 - 0:30 | Part 1. Least squares three ways |
| 0:30 - 0:50 | Part 2. The geometry of least squares |
| 0:50 - 1:10 | Part 3. Sampling properties |
| 1:10 - 1:30 | Part 4. Basis expansions and overfitting |
| 1:30 - 1:45 | Part 5. Maximum likelihood |
| 1:45 - 2:00 | Part 6. Bayesian regression, and Part 7 on real data |

Notation follows the lecture: $n$ observations indexed by $i = 1, \dots, n$, $p$ predictors, design matrix $\mathbf{X}$, coefficient vector $\boldsymbol{\beta}$.

## Setup

**If you are running this on Google Colab, save your own copy now**: `File` &rarr; `Save a copy in Drive`. Otherwise everything you write during the next two hours disappears when you close the tab.

The cell below downloads the course helper module if it is not already present, then imports everything the lab needs. It works identically on Colab and on a local Anaconda installation.

In [ ]:
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/korobilis/ECON5129-labs/main"

if not os.path.exists("econ5129_utils.py"):
    urllib.request.urlretrieve(f"{REPO_RAW}/econ5129_utils.py", "econ5129_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, stats

import econ5129_utils as e5

e5.set_style()
rng = np.random.default_rng(5129)

print(f"numpy {np.__version__}, pandas {pd.__version__}")

## Part 1. Least squares three ways

The linear model is

$$ y_i = \mathbf{x}_i' \boldsymbol{\beta} + \varepsilon_i, \qquad i = 1, \dots, n, $$

or in matrix form $\mathbf{y} = \mathbf{X} \boldsymbol{\beta} + \boldsymbol{\varepsilon}$. Least squares minimises the residual sum of squares and solves the normal equations

$$ \mathbf{X}'\mathbf{X} \widehat{\boldsymbol{\beta}} = \mathbf{X}'\mathbf{y}. $$

We start with simulated data, so that we know the answer we are trying to recover.

In [ ]:
n, p = 200, 3
beta_true = np.array([1.0, -2.0, 0.5, 3.0])   # intercept first
sigma = 1.0

X_raw = rng.normal(size=(n, p))
X = np.column_stack([np.ones(n), X_raw])       # add the intercept column
y = X @ beta_true + sigma * rng.normal(size=n)

print(f"X has shape {X.shape}, y has shape {y.shape}")

Three routes to the same estimator.

**Route 1: solve the normal equations.** This forms $\mathbf{X}'\mathbf{X}$ and $\mathbf{X}'\mathbf{y}$ and asks a linear solver for the answer. Note that we never invert anything.

In [ ]:
XtX = X.T @ X
Xty = X.T @ y
beta_normal = np.linalg.solve(XtX, Xty)

**Route 2: least squares directly.** `np.linalg.lstsq` works from a QR decomposition of $\mathbf{X}$, so it never forms $\mathbf{X}'\mathbf{X}$ at all. This matters because squaring a matrix squares its condition number.

In [ ]:
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

**Route 3: scikit-learn.** The interface you will use for the rest of the course. `fit_intercept=False` because our design matrix already contains a column of ones.

In [ ]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression(fit_intercept=False).fit(X, y)
beta_sklearn = ols.coef_

In [ ]:
comparison = pd.DataFrame(
    {
        "true": beta_true,
        "normal equations": beta_normal,
        "lstsq (QR)": beta_lstsq,
        "scikit-learn": beta_sklearn,
    },
    index=["intercept", "x1", "x2", "x3"],
)
comparison.round(6)

All three agree to printing precision, and all three are close to but not equal to the truth. That gap is estimation error, and quantifying it is the subject of Part 3.

### Why you should never write `inv(X'X)`

The formula $\widehat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}$ is how the estimator is *defined*, not how it should be *computed*. Explicit inversion is slower and numerically worse. The following design is nearly collinear, which is exactly the situation where the difference shows up.

In [ ]:
z1 = rng.normal(size=n)
z2 = z1 + 1e-7 * rng.normal(size=n)          # almost identical to z1
X_ill = np.column_stack([np.ones(n), z1, z2])
y_ill = X_ill @ np.array([1.0, 2.0, -1.0]) + 0.1 * rng.normal(size=n)

beta_inv = np.linalg.inv(X_ill.T @ X_ill) @ (X_ill.T @ y_ill)
beta_qr, *_ = np.linalg.lstsq(X_ill, y_ill, rcond=None)

print(f"condition number of X      : {np.linalg.cond(X_ill):.3e}")
print(f"condition number of X'X    : {np.linalg.cond(X_ill.T @ X_ill):.3e}")
print(f"residual sum of squares, inv: {np.sum((y_ill - X_ill @ beta_inv) ** 2):.8f}")
print(f"residual sum of squares, QR : {np.sum((y_ill - X_ill @ beta_qr) ** 2):.8f}")

The condition number of $\mathbf{X}'\mathbf{X}$ is the square of the condition number of $\mathbf{X}$: forming the cross-product matrix throws away half of your available precision. Use `solve` or `lstsq`, never `inv`.

### Exercise 1

Write a function `ols(X, y)` that returns a dictionary containing the coefficient vector, the residual vector, the unbiased error variance estimate

$$ s^2 = \frac{1}{n - p} \sum_{i=1}^{n} \widehat{\varepsilon}_i^2 $$

(where $p$ counts the intercept), the standard errors $\sqrt{s^2 [(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}$, the $t$ statistics, and $R^2$. Use it on the simulated data and check that the true coefficients lie inside the implied 95% confidence intervals.

In [ ]:
def ols(X, y):  #@keep
    """Least squares estimation with standard errors."""  #@keep
    n, p = X.shape
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    s2 = resid @ resid / (n - p)
    XtX_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(s2 * np.diag(XtX_inv))
    tss = np.sum((y - y.mean()) ** 2)
    return {
        "beta": beta,
        "resid": resid,
        "s2": s2,
        "se": se,
        "t": beta / se,
        "r2": 1.0 - resid @ resid / tss,
    }


fit = ols(X, y)

results = pd.DataFrame(
    {
        "estimate": fit["beta"],
        "std error": fit["se"],
        "t stat": fit["t"],
        "lower 95%": fit["beta"] - 1.96 * fit["se"],
        "upper 95%": fit["beta"] + 1.96 * fit["se"],
        "true": beta_true,
    },
    index=["intercept", "x1", "x2", "x3"],
)
print(results.round(3))
print(f"\nR-squared: {fit['r2']:.4f}, s2: {fit['s2']:.4f} (true sigma^2 = {sigma ** 2})")

## Part 2. The geometry of least squares

Least squares projects $\mathbf{y}$ orthogonally onto the column space of $\mathbf{X}$. The projection is carried out by the hat matrix

$$ \mathbf{H} = \mathbf{X}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}', \qquad \widehat{\mathbf{y}} = \mathbf{H}\mathbf{y}, \qquad \widehat{\boldsymbol{\varepsilon}} = (\mathbf{I} - \mathbf{H})\mathbf{y}. $$

Everything in the lecture's geometric section is a property of $\mathbf{H}$, and every one of them can be checked numerically.

In [ ]:
H = X @ np.linalg.solve(X.T @ X, X.T)
y_hat = H @ y
resid = y - y_hat

print(f"H is symmetric      : {np.allclose(H, H.T)}")
print(f"H is idempotent     : {np.allclose(H @ H, H)}")
print(f"trace(H) = {np.trace(H):.6f}, and p = {X.shape[1]}")
print(f"residuals orthogonal to X: max |X'e| = {np.abs(X.T @ resid).max():.2e}")
print(f"residuals orthogonal to fitted values: {y_hat @ resid:.2e}")

The trace of an idempotent matrix equals its rank, so $\operatorname{tr}(\mathbf{H}) = p$ counts the dimensions of the space we project onto. This is the sense in which a model has "$p$ degrees of freedom", and in Lecture 5 we will meet smoothers whose effective degrees of freedom are not integers.

### $R^2$ is a squared cosine

Write $\tilde{\mathbf{y}}$ for the demeaned response. Then $R^2 = \|\widehat{\tilde{\mathbf{y}}}\|^2 / \|\tilde{\mathbf{y}}\|^2$, which is exactly $\cos^2 \theta$ where $\theta$ is the angle between the demeaned response and its projection.

In [ ]:
y_tilde = y - y.mean()
y_hat_tilde = y_hat - y_hat.mean()

r2 = np.sum(y_hat_tilde ** 2) / np.sum(y_tilde ** 2)
cos_theta = (y_tilde @ y_hat_tilde) / (np.linalg.norm(y_tilde) * np.linalg.norm(y_hat_tilde))

print(f"R-squared     : {r2:.6f}")
print(f"cos^2(theta)  : {cos_theta ** 2:.6f}")
print(f"angle theta   : {np.degrees(np.arccos(cos_theta)):.2f} degrees")

### Exercise 2

The diagonal element $h_{ii}$ of the hat matrix is the **leverage** of observation $i$: it measures how strongly that single observation pulls its own fitted value.

1. Extract the leverages and confirm that they sum to $p$ and that each lies in $[0, 1]$.
2. Plot the leverages against the observation index and mark the average value $p/n$.
3. Now append a single outlying observation whose predictors are far from the rest of the sample, refit, and report its leverage.

What does this tell you about the influence a single observation can have on a least squares fit?

In [ ]:
lev = np.diag(H)  #@keep

print(f"sum of leverages: {lev.sum():.6f} (p = {X.shape[1]})")
print(f"range: [{lev.min():.4f}, {lev.max():.4f}], average p/n = {X.shape[1] / n:.4f}")

# add an outlying observation in predictor space
x_out = np.array([1.0, 6.0, -6.0, 6.0])
X_aug = np.vstack([X, x_out])
y_aug = np.append(y, x_out @ beta_true)

H_aug = X_aug @ np.linalg.solve(X_aug.T @ X_aug, X_aug.T)
lev_aug = np.diag(H_aug)

fig, ax = plt.subplots()
ax.stem(lev_aug, markerfmt=" ", basefmt=" ")
ax.axhline(X.shape[1] / (n + 1), color=e5.COLORS[2], ls="--", label="average $p/n$")
ax.set_xlabel("observation")
ax.set_ylabel("leverage $h_{ii}$")
ax.set_title("Leverage, with one outlying observation appended")
ax.legend()
plt.show()

print(f"leverage of the appended observation: {lev_aug[-1]:.4f}")

## Part 3. Sampling properties

Under the classical assumptions,

$$ \mathbb{E}[\widehat{\boldsymbol{\beta}}] = \boldsymbol{\beta}, \qquad \operatorname{Var}(\widehat{\boldsymbol{\beta}}) = \sigma^2 (\mathbf{X}'\mathbf{X})^{-1}. $$

These are statements about repeated samples. Since we control the data generating process, we can draw those repeated samples and check.

In [ ]:
n_rep = 2000
X_fixed = np.column_stack([np.ones(n), rng.normal(size=(n, p))])
beta_draws = np.empty((n_rep, p + 1))

for r in range(n_rep):
    y_r = X_fixed @ beta_true + sigma * rng.normal(size=n)
    beta_draws[r] = np.linalg.solve(X_fixed.T @ X_fixed, X_fixed.T @ y_r)

var_theory = sigma ** 2 * np.linalg.inv(X_fixed.T @ X_fixed)

summary = pd.DataFrame(
    {
        "true": beta_true,
        "Monte Carlo mean": beta_draws.mean(axis=0),
        "Monte Carlo sd": beta_draws.std(axis=0, ddof=1),
        "theoretical sd": np.sqrt(np.diag(var_theory)),
    },
    index=["intercept", "x1", "x2", "x3"],
)
summary.round(4)

In [ ]:
fig, ax = plt.subplots()
ax.hist(beta_draws[:, 1], bins=40, density=True, alpha=0.6,
        color=e5.COLORS[1], edgecolor="white")
grid = np.linspace(beta_draws[:, 1].min(), beta_draws[:, 1].max(), 300)
ax.plot(grid, stats.norm.pdf(grid, beta_true[1], np.sqrt(var_theory[1, 1])),
        color=e5.COLORS[0], label="theoretical density")
ax.axvline(beta_true[1], color=e5.COLORS[2], ls="--", label="true value")
ax.set_xlabel(r"$\widehat{\beta}_1$")
ax.set_ylabel("density")
ax.set_title("Sampling distribution of the slope over 2000 samples")
ax.legend()
plt.show()

### Where the variance comes from

Write the singular value decomposition $\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}'$ with singular values $d_1 \ge \dots \ge d_p > 0$. Then

$$ \operatorname{Var}(\widehat{\boldsymbol{\beta}}) = \sigma^2 (\mathbf{X}'\mathbf{X})^{-1} = \sigma^2 \sum_{j=1}^{p} \frac{1}{d_j^2} \mathbf{v}_j \mathbf{v}_j'. $$

Directions in predictor space with little variation, meaning small $d_j$, contribute enormous variance. This single expression is the reason shrinkage exists, and we will return to it in Lecture 3.

In [ ]:
U, d, Vt = np.linalg.svd(X_fixed, full_matrices=False)

contrib = sigma ** 2 / d ** 2
svd_table = pd.DataFrame(
    {
        "singular value $d_j$": d,
        "variance contribution $\\sigma^2 / d_j^2$": contrib,
        "share of total": contrib / contrib.sum(),
    }
)
svd_table.round(6)

### Exercise 3

Rebuild the design matrix so that the three predictors are strongly correlated, using

$$ \Sigma_{jk} = \rho^{|j - k|}, \qquad \rho = 0.95, $$

and repeat the Monte Carlo experiment.

1. By what factor does the standard deviation of $\widehat{\beta}_1$ increase relative to the uncorrelated design?
2. Recompute the singular values. Which direction now dominates the variance?
3. Are the estimates still unbiased?

In [ ]:
rho = 0.95  #@keep
idx = np.arange(p)  #@keep
Sigma = rho ** np.abs(idx[:, None] - idx[None, :])  #@keep
L = np.linalg.cholesky(Sigma)

X_corr = np.column_stack([np.ones(n), rng.normal(size=(n, p)) @ L.T])

beta_corr = np.empty((n_rep, p + 1))
for r in range(n_rep):
    y_r = X_corr @ beta_true + sigma * rng.normal(size=n)
    beta_corr[r] = np.linalg.solve(X_corr.T @ X_corr, X_corr.T @ y_r)

sd_indep = beta_draws.std(axis=0, ddof=1)
sd_corr = beta_corr.std(axis=0, ddof=1)

print(pd.DataFrame(
    {
        "sd, independent": sd_indep,
        "sd, rho = 0.95": sd_corr,
        "inflation factor": sd_corr / sd_indep,
        "bias": beta_corr.mean(axis=0) - beta_true,
    },
    index=["intercept", "x1", "x2", "x3"],
).round(4))

d_corr = np.linalg.svd(X_corr, compute_uv=False)
print(f"\nsingular values, independent design: {np.round(d, 2)}")
print(f"singular values, correlated design : {np.round(d_corr, 2)}")

The estimator remains unbiased, exactly as Gauss-Markov promises. What collapses is precision. Unbiasedness on its own is close to worthless for prediction, which is the opening argument of Lecture 3.

## Part 4. Basis expansions and overfitting

"Linear regression" means linear in $\boldsymbol{\beta}$, not linear in $x$. Replacing $x$ with a vector of transformations $\boldsymbol{\phi}(x) = (\phi_1(x), \dots, \phi_M(x))'$ leaves every formula above intact while allowing arbitrary curvature.

The underlying function below is $f(x) = \sin(2\pi x)$, which no polynomial reproduces exactly.

In [ ]:
def f_true(x):
    return np.sin(2 * np.pi * x)


n_train, n_test, noise = 40, 400, 0.25

x_train = np.sort(rng.uniform(0, 1, n_train))
y_train = f_true(x_train) + noise * rng.normal(size=n_train)

x_test = np.sort(rng.uniform(0, 1, n_test))
y_test = f_true(x_test) + noise * rng.normal(size=n_test)


def poly_basis(x, degree):
    """Design matrix of monomials up to the given degree, intercept included."""
    return np.vander(x, degree + 1, increasing=True)

In [ ]:
degrees = [1, 3, 9, 15]
grid = np.linspace(0, 1, 400)

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, deg in zip(axes.ravel(), degrees):
    b, *_ = np.linalg.lstsq(poly_basis(x_train, deg), y_train, rcond=None)
    ax.scatter(x_train, y_train, s=18, color=e5.COLORS[6], zorder=3, label="training data")
    ax.plot(grid, f_true(grid), color=e5.COLORS[0], label="truth")
    ax.plot(grid, poly_basis(grid, deg) @ b, color=e5.COLORS[2], label="fit")
    ax.set_title(f"degree {deg}")
    ax.set_ylim(-2, 2)
axes[0, 0].legend(fontsize=8)
fig.suptitle("Polynomial basis expansions of increasing flexibility")
fig.tight_layout()
plt.show()

The degree 15 fit passes very close to the training points and behaves badly between them. To make that precise, track training and test error as a function of the degree.

In [ ]:
max_deg = 18
train_mse, test_mse = [], []

for deg in range(1, max_deg + 1):
    b, *_ = np.linalg.lstsq(poly_basis(x_train, deg), y_train, rcond=None)
    train_mse.append(e5.mse(y_train, poly_basis(x_train, deg) @ b))
    test_mse.append(e5.mse(y_test, poly_basis(x_test, deg) @ b))

fig, ax = plt.subplots()
ax.plot(range(1, max_deg + 1), train_mse, marker="o", ms=4, label="training MSE")
ax.plot(range(1, max_deg + 1), test_mse, marker="s", ms=4, label="test MSE")
ax.axhline(noise ** 2, color=e5.COLORS[6], ls=":", label="irreducible error")
ax.axvline(int(np.argmin(test_mse)) + 1, color=e5.COLORS[2], ls="--",
           label=f"best degree = {int(np.argmin(test_mse)) + 1}")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("mean squared error")
ax.set_title("Training error falls monotonically; test error does not")
ax.legend()
plt.show()

### Exercise 4

Polynomials are global: every coefficient affects the fit everywhere, so what happens at one end of the sample moves the curve at the other end. The standard alternative is a **local** basis, in which each function is active only in a neighbourhood. Gaussian radial basis functions are the simplest example,

$$ \phi_m(x) = \exp\left\{ -\frac{(x - c_m)^2}{2 s^2} \right\}, \qquad m = 1, \dots, M, $$

with centres $c_m$ spread evenly over the range of $x$ and a common width $s$.

1. Write `rbf_basis(x, centres, width)` returning the design matrix with an intercept column.
2. Reproduce the training and test error curve, now as a function of the number of centres $M$, holding the width at $s = 1/M$.
3. Compare the best radial basis fit to the best polynomial fit, both in test error and visually near the boundaries of the sample.

The two will achieve similar test error on this problem. Look at where they differ rather than at the headline number.

In [ ]:
def rbf_basis(x, centres, width):  #@keep
    """Gaussian radial basis design matrix with an intercept column."""  #@keep
    Z = np.exp(-0.5 * ((x[:, None] - centres[None, :]) / width) ** 2)
    return np.column_stack([np.ones(len(x)), Z])


M_grid = range(2, 21)
train_rbf, test_rbf = [], []

for M in M_grid:
    centres = np.linspace(0, 1, M)
    width = 1.0 / M
    B_train = rbf_basis(x_train, centres, width)
    b, *_ = np.linalg.lstsq(B_train, y_train, rcond=None)
    train_rbf.append(e5.mse(y_train, B_train @ b))
    test_rbf.append(e5.mse(y_test, rbf_basis(x_test, centres, width) @ b))

best_M = list(M_grid)[int(np.argmin(test_rbf))]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(M_grid), train_rbf, marker="o", ms=4, label="training MSE")
axes[0].plot(list(M_grid), test_rbf, marker="s", ms=4, label="test MSE")
axes[0].axvline(best_M, color=e5.COLORS[2], ls="--", label=f"best $M$ = {best_M}")
axes[0].set_yscale("log")
axes[0].set_xlabel("number of basis functions $M$")
axes[0].set_ylabel("mean squared error")
axes[0].legend()

centres = np.linspace(0, 1, best_M)
b_rbf, *_ = np.linalg.lstsq(rbf_basis(x_train, centres, 1.0 / best_M), y_train, rcond=None)
best_deg = int(np.argmin(test_mse)) + 1
b_poly, *_ = np.linalg.lstsq(poly_basis(x_train, best_deg), y_train, rcond=None)

axes[1].scatter(x_train, y_train, s=18, color=e5.COLORS[6], zorder=3)
axes[1].plot(grid, f_true(grid), color=e5.COLORS[0], label="truth")
axes[1].plot(grid, rbf_basis(grid, centres, 1.0 / best_M) @ b_rbf,
             color=e5.COLORS[2], label=f"radial basis, $M$ = {best_M}")
axes[1].plot(grid, poly_basis(grid, best_deg) @ b_poly, color=e5.COLORS[4],
             ls="--", label=f"polynomial, degree {best_deg}")
axes[1].set_ylim(-2, 2)
axes[1].legend(fontsize=9)
fig.suptitle("Local basis functions against global polynomials")
fig.tight_layout()
plt.show()

print(f"best test MSE, polynomial   : {min(test_mse):.4f}")
print(f"best test MSE, radial basis : {min(test_rbf):.4f}")

## Part 5. Maximum likelihood

Adding the assumption $\varepsilon_i \sim N(0, \sigma^2)$ turns the model into a probability statement, with log-likelihood

$$ \ell(\boldsymbol{\beta}, \sigma^2) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(y_i - \mathbf{x}_i'\boldsymbol{\beta})^2 . $$

Maximising over $\boldsymbol{\beta}$ is the same as minimising the sum of squares, so the maximum likelihood estimator of $\boldsymbol{\beta}$ *is* the least squares estimator. Nothing forces us to accept that conclusion on faith: we can optimise numerically and see.

We parametrise by $\log \sigma$ so that the optimiser can roam over the whole real line.

In [ ]:
def neg_loglik(theta, X, y):
    """Negative Gaussian log-likelihood; theta = (beta, log sigma)."""
    beta, log_sigma = theta[:-1], theta[-1]
    sigma = np.exp(log_sigma)
    resid = y - X @ beta
    n = len(y)
    return 0.5 * n * np.log(2 * np.pi) + n * log_sigma + 0.5 * resid @ resid / sigma ** 2


start = np.zeros(X.shape[1] + 1)
opt = optimize.minimize(neg_loglik, start, args=(X, y), method="BFGS")

beta_mle = opt.x[:-1]
sigma2_mle = np.exp(2 * opt.x[-1])

print(pd.DataFrame(
    {"least squares": beta_normal, "maximum likelihood": beta_mle, "true": beta_true},
    index=["intercept", "x1", "x2", "x3"],
).round(6))

resid_ols = y - X @ beta_normal
s2_unbiased = resid_ols @ resid_ols / (n - X.shape[1])

print(f"\nMLE of sigma^2       : {sigma2_mle:.4f}")
print(f"unbiased estimate s2 : {s2_unbiased:.4f}")
print(f"ratio                : {sigma2_mle / s2_unbiased:.4f}  vs  (n - p)/n = {(n - X.shape[1]) / n:.4f}")

The coefficients coincide. The variance estimates do not: the maximum likelihood estimator of $\sigma^2$ divides by $n$ rather than $n - p$, and is therefore biased downwards by exactly the factor $(n-p)/n$.

### Exercise 5

The choice of likelihood is a modelling choice, and changing it changes the estimator. Replace the Gaussian with a Student $t$ likelihood with $\nu = 3$ degrees of freedom,

$$ \ell(\boldsymbol{\beta}, \sigma) = \sum_{i=1}^{n} \log t_\nu\!\left( \frac{y_i - \mathbf{x}_i'\boldsymbol{\beta}}{\sigma} \right) - n \log \sigma . $$

1. Contaminate the data: replace five randomly chosen responses with $y_i + 15$.
2. Estimate by least squares and by Student $t$ maximum likelihood.
3. Compare both to the truth, and explain the difference in terms of how each objective function weights large residuals.

`stats.t.logpdf` will give you the log density.

In [ ]:
y_contam = y.copy()  #@keep
outliers = rng.choice(n, size=5, replace=False)  #@keep
y_contam[outliers] += 15.0  #@keep


def neg_loglik_t(theta, X, y, nu=3.0):
    beta, log_sigma = theta[:-1], theta[-1]
    sigma = np.exp(log_sigma)
    resid = (y - X @ beta) / sigma
    return -(np.sum(stats.t.logpdf(resid, df=nu)) - len(y) * log_sigma)


beta_ols_contam = np.linalg.solve(X.T @ X, X.T @ y_contam)
opt_t = optimize.minimize(neg_loglik_t, np.zeros(X.shape[1] + 1),
                          args=(X, y_contam), method="BFGS")
beta_t = opt_t.x[:-1]

print(pd.DataFrame(
    {
        "true": beta_true,
        "least squares": beta_ols_contam,
        "Student t MLE": beta_t,
    },
    index=["intercept", "x1", "x2", "x3"],
).round(3))

r = np.linspace(-6, 6, 400)
fig, ax = plt.subplots()
ax.plot(r, 0.5 * r ** 2, color=e5.COLORS[0], label="Gaussian: quadratic loss")
ax.plot(r, -stats.t.logpdf(r, df=3) + stats.t.logpdf(0, df=3),
        color=e5.COLORS[2], label="Student $t_3$: logarithmic loss")
ax.set_xlabel("residual")
ax.set_ylabel("contribution to the objective")
ax.set_title("Why the Student $t$ likelihood ignores outliers")
ax.legend()
plt.show()

## Part 6. Bayesian regression

Place a prior $\boldsymbol{\beta} \sim N(\mathbf{0}, \tau^2 \mathbf{I})$ and treat $\sigma^2$ as known. Combining prior and likelihood gives a Gaussian posterior with

$$ \mathbf{V}_{\text{post}} = \left( \frac{\mathbf{X}'\mathbf{X}}{\sigma^2} + \frac{\mathbf{I}}{\tau^2} \right)^{-1}, \qquad \boldsymbol{\mu}_{\text{post}} = \mathbf{V}_{\text{post}} \frac{\mathbf{X}'\mathbf{y}}{\sigma^2}. $$

The posterior mean, which is also the MAP estimate under a Gaussian posterior, is algebraically identical to ridge regression with $\lambda = \sigma^2/\tau^2$. Regularisation and prior information are the same statement in two languages.

To make the prior bite, we use a small sample.

In [ ]:
n_small = 25
X_small = np.column_stack([np.ones(n_small), rng.normal(size=(n_small, p))])
y_small = X_small @ beta_true + sigma * rng.normal(size=n_small)


def bayes_regression(X, y, tau2, sigma2):
    """Posterior mean and covariance under a N(0, tau2 I) prior."""
    prec = X.T @ X / sigma2 + np.eye(X.shape[1]) / tau2
    V = np.linalg.inv(prec)
    mu = V @ (X.T @ y) / sigma2
    return mu, V


tau2 = 1.0
mu_post, V_post = bayes_regression(X_small, y_small, tau2, sigma ** 2)

lam = sigma ** 2 / tau2
beta_ridge = np.linalg.solve(X_small.T @ X_small + lam * np.eye(X_small.shape[1]),
                             X_small.T @ y_small)

print(pd.DataFrame(
    {
        "true": beta_true,
        "least squares": np.linalg.solve(X_small.T @ X_small, X_small.T @ y_small),
        "posterior mean": mu_post,
        f"ridge, lambda = {lam:g}": beta_ridge,
        "posterior sd": np.sqrt(np.diag(V_post)),
    },
    index=["intercept", "x1", "x2", "x3"],
).round(4))

### The shrinkage path

As $\tau^2 \to \infty$ the prior becomes uninformative and the posterior mean converges to least squares. As $\tau^2 \to 0$ every coefficient is pulled to zero. The path between the two is the ridge path of Lecture 3, viewed from the Bayesian side.

In [ ]:
tau2_grid = np.logspace(-3, 3, 60)
paths = np.array([bayes_regression(X_small, y_small, t, sigma ** 2)[0] for t in tau2_grid])

fig, ax = plt.subplots()
for j, name in enumerate(["intercept", "x1", "x2", "x3"]):
    ax.plot(tau2_grid, paths[:, j], label=name)
    ax.axhline(beta_true[j], color=e5.COLORS[j], ls=":", lw=1)
ax.set_xscale("log")
ax.set_xlabel(r"prior variance $\tau^2$")
ax.set_ylabel("posterior mean")
ax.set_title("From the prior mean to least squares as the prior weakens")
ax.legend(ncol=4, fontsize=9)
plt.show()

### Predictive distribution

The object a forecaster actually wants is not the coefficient but the predictive distribution of a future observation. Integrating the likelihood over the posterior gives

$$ y_{n+1} \mid \mathbf{y} \sim N\!\left( \mathbf{x}_{n+1}' \boldsymbol{\mu}_{\text{post}}, \; \mathbf{x}_{n+1}' \mathbf{V}_{\text{post}} \mathbf{x}_{n+1} + \sigma^2 \right). $$

The two variance terms separate parameter uncertainty from irreducible noise. We illustrate this with a single predictor and a basis expansion, so the bands can be drawn.

In [ ]:
x_b = np.sort(rng.uniform(0, 1, 20))
y_b = f_true(x_b) + noise * rng.normal(size=20)

centres_b = np.linspace(0, 1, 8)
width_b = 0.12


def rbf(x):
    Z = np.exp(-0.5 * ((x[:, None] - centres_b[None, :]) / width_b) ** 2)
    return np.column_stack([np.ones(len(x)), Z])


mu_b, V_b = bayes_regression(rbf(x_b), y_b, tau2=1.0, sigma2=noise ** 2)

B_grid = rbf(grid)
mean_pred = B_grid @ mu_b
var_param = np.einsum("ij,jk,ik->i", B_grid, V_b, B_grid)
sd_pred = np.sqrt(var_param + noise ** 2)

fig, ax = plt.subplots()
ax.fill_between(grid, mean_pred - 1.96 * sd_pred, mean_pred + 1.96 * sd_pred,
                color=e5.COLORS[1], alpha=0.25, label="95% predictive interval")
ax.fill_between(grid, mean_pred - 1.96 * np.sqrt(var_param),
                mean_pred + 1.96 * np.sqrt(var_param),
                color=e5.COLORS[1], alpha=0.45, label="parameter uncertainty only")
ax.plot(grid, mean_pred, color=e5.COLORS[0], label="posterior mean")
ax.plot(grid, f_true(grid), color=e5.COLORS[2], ls="--", label="truth")
ax.scatter(x_b, y_b, s=25, color=e5.COLORS[6], zorder=3, label="data")
ax.set_title("Bayesian basis regression with predictive uncertainty")
ax.legend(fontsize=9, loc="lower left")
plt.show()

Notice that the bands widen where the data are sparse. A point estimate cannot tell you that, and this is precisely the property that makes Gaussian processes attractive in Lecture 5.

### Exercise 6

Assess whether the predictive intervals are honest.

1. Simulate 500 fresh test points from the same process.
2. Compute the fraction that fall inside the 95% predictive interval. Is it close to 0.95?
3. Repeat with a badly chosen prior, $\tau^2 = 0.01$. What happens to coverage, and why?

In [ ]:
def coverage(tau2_value, n_new=500):  #@keep
    mu_c, V_c = bayes_regression(rbf(x_b), y_b, tau2_value, noise ** 2)
    x_new = rng.uniform(0, 1, n_new)
    y_new = f_true(x_new) + noise * rng.normal(size=n_new)

    B_new = rbf(x_new)
    m = B_new @ mu_c
    s = np.sqrt(np.einsum("ij,jk,ik->i", B_new, V_c, B_new) + noise ** 2)
    inside = np.abs(y_new - m) <= 1.96 * s
    return inside.mean(), np.mean((y_new - m) ** 2)


for t in [1.0, 0.01]:
    cov, msep = coverage(t)
    print(f"tau^2 = {t:>5}: coverage = {cov:.3f}, predictive MSE = {msep:.4f}")

## Part 7. A first look at real data

We close with the FRED-MD panel, the macroeconomic dataset that will run through most of the remaining labs. It contains 126 monthly US series from 1959 onwards, together with a transformation code for each series that renders it stationary.

The target is monthly industrial production growth. The predictors are a small, deliberately conventional set.

In [ ]:
data, codes = e5.load_fredmd()
print(f"{data.shape[0]} months from {data.index[0]:%Y-%m} to {data.index[-1]:%Y-%m}, "
      f"{data.shape[1]} series")

transformed = e5.fredmd_transform(data, codes)

predictors = ["UNRATE", "FEDFUNDS", "S&P 500", "CPIAUCSL", "CLAIMSx", "T10YFFM"]
panel = transformed[["INDPRO"] + predictors].dropna()
panel = panel.loc["1960-01":"2019-12"]

y_macro = 100 * panel["INDPRO"].to_numpy()
X_macro = np.column_stack([np.ones(len(panel)), panel[predictors].to_numpy()])

n_macro, p_macro = X_macro.shape
beta_macro = np.linalg.solve(X_macro.T @ X_macro, X_macro.T @ y_macro)
resid_macro = y_macro - X_macro @ beta_macro
s2_macro = resid_macro @ resid_macro / (n_macro - p_macro)
se_macro = np.sqrt(s2_macro * np.diag(np.linalg.inv(X_macro.T @ X_macro)))
r2_macro = 1.0 - resid_macro @ resid_macro / np.sum((y_macro - y_macro.mean()) ** 2)

print(pd.DataFrame(
    {"estimate": beta_macro, "std error": se_macro, "t stat": beta_macro / se_macro},
    index=["intercept"] + predictors,
).round(4))
print(f"\nn = {n_macro},  R-squared = {r2_macro:.4f}")

Two features of that table are worth pausing on. The intercept recovers average monthly industrial production growth of roughly 0.18%. The coefficient on equity returns is insignificant and carries the wrong sign, which is routine in a correlated design and a first warning that individual coefficients are not the same object as a good prediction.

An $R^2$ of this size is entirely normal for monthly growth rates: most month-to-month movement in industrial production is unforecastable. Keep that number in mind, because for the rest of the course the question is never "is the fit good" but "is it better than the benchmark, out of sample".

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(panel.index, y_macro, color=e5.COLORS[0], lw=1.0, label="actual")
axes[0].plot(panel.index, X_macro @ beta_macro, color=e5.COLORS[2], lw=1.2, label="fitted")
axes[0].set_ylabel("IP growth, % per month")
axes[0].legend()

axes[1].plot(panel.index, resid_macro, color=e5.COLORS[6], lw=0.9)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_ylabel("residual")
axes[1].set_xlabel("")
fig.suptitle("Industrial production growth: in-sample fit and residuals")
fig.tight_layout()
plt.show()

### Exercise 7

The residual plot shows visible clustering of large residuals. Overlay the NBER recession indicator, available through `e5.load_usrec()`, and check whether the model's errors are systematically larger in recessions.

1. Load the indicator and align it to the sample.
2. Compare the root mean squared residual in recession months against expansion months.
3. Shade the recession periods on the residual plot.

What does this suggest about the assumption of constant error variance, and about what a linear model with fixed coefficients can capture?

In [ ]:
rec = e5.load_usrec().reindex(panel.index)  #@keep
in_rec = rec.to_numpy() == 1.0  #@keep

print(f"months in recession : {in_rec.sum()} of {len(in_rec)}")
print(f"RMSE in recessions  : {np.sqrt(np.mean(resid_macro[in_rec] ** 2)):.4f}")
print(f"RMSE in expansions  : {np.sqrt(np.mean(resid_macro[~in_rec] ** 2)):.4f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(panel.index, resid_macro, color=e5.COLORS[0], lw=0.9)
ax.axhline(0, color="black", lw=0.8)
ax.fill_between(panel.index, resid_macro.min(), resid_macro.max(), where=in_rec,
                color=e5.COLORS[2], alpha=0.15, label="NBER recession")
ax.set_ylabel("residual")
ax.set_title("Errors are far larger in recessions")
ax.legend()
plt.show()

## Take-home challenges

1. **Frisch-Waugh-Lovell.** Regress `INDPRO` growth on `UNRATE` alone after partialling out the remaining predictors from both, and verify that you recover exactly the multiple regression coefficient on `UNRATE`. Explain what this says about what a regression coefficient measures.

2. **Degrees of freedom.** For the radial basis fit, compute $\operatorname{tr}(\mathbf{H})$ for each $M$ and plot test error against effective degrees of freedom rather than against $M$. Where does the minimum sit relative to $n$?

3. **Ridge as data augmentation.** Show numerically that ridge regression with penalty $\lambda$ is ordinary least squares applied to the augmented data $\tilde{\mathbf{X}} = [\mathbf{X}; \sqrt{\lambda}\mathbf{I}]$ and $\tilde{\mathbf{y}} = [\mathbf{y}; \mathbf{0}]$. Relate the augmented rows to the prior of Part 6.

4. **Prior sensitivity.** Repeat Part 6 with a prior centred away from zero, $\boldsymbol{\beta} \sim N(\mathbf{b}_0, \tau^2\mathbf{I})$ with $\mathbf{b}_0 = (0, 1, 1, 1)'$. Derive the modified posterior mean, implement it, and show how quickly the data overwhelm a wrong prior as $n$ grows.

---

**Next**: Lecture 2 turns from predicting a number to predicting a label. Lab 2 builds the Bayes classifier, $k$-nearest neighbours, discriminant analysis and logistic regression, and applies them to dating US recessions.